In [2]:
import osmnx as ox
import pandas as pd 
import geopandas as gpd
import matplotlib.pyplot as plt 
from shapely.geometry import box
from sklearn.neighbors import BallTree

import seaborn as sns
import numpy as np
import time
from shapely.geometry import Point

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


In [3]:
#importing requirement dataset

#Dataset Point of Interest
df_poi = pd.read_csv('data/osmnx_data/data_titik_poi_lengkap_jabar.csv')
df_poi.head()

,lat,lon,kategori
0,-6.914742,107.614526,sekolah
1,-6.896498,107.589996,sekolah
2,-6.865456,107.605989,sekolah
3,-6.875213,107.605027,sekolah
4,-7.275235,107.105825,sekolah


In [ ]:
#Fungsi untuk mengambil alamat tergantung lattitude dan longitude

geolocator = Nominatim(user_agent="my_request")

reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_address(lat, lon):
    try:
        location = reverse((lat,lon))
        return location.address
    except:
        return None

df_poi['alamat'] = df_poi.apply(lambda x: get_address(x['lat'], x['lon']), axis=1)
df_poi.to_csv("data_titik_poi_lengkap_jabar_alamat.csv", index=False)

In [4]:
#Proses pembuatan data koordinat  untuk wilayah Jawa Barat
#INI JANGAN DI RUN SEMBARANGAN!
area_name = "Kota Bandung, Jawa Barat, Indonesia"
polygon_bandung = ox.geocode_to_gdf(area_name).geometry.iloc[0]

#Dictionary buat kategori yang mau diunduh datanya dari osm 
categories = {
    "sekolah": {"amenity": ["school"]},
    "restoran": {"amenity": ["restaurant"]},
    "pasar": {"amenity": ["marketplace"]},
    "kampus": {"amenity": ["college", "university"]},
    "halte_bus": {"amenity": ["bus_station"]},
    "kafe": {"amenity": ["cafe"]},
}

# Fungsi untuk mengunduh data poi berdasarkan kategori dan area dan mengambil longitude dan latitude dari centroid geometry
def get_poi_data(categories_dict, polygon_area):
    all_results = {}

    for name, tags in categories_dict.items():
        try:
            print(f"Lagi ngunduh data {name}..." )

            poi = ox.features_from_polygon(polygon=polygon_area, tags=tags)
            
            if not poi.empty:
                poi = poi[poi.geometry.notnull()].copy()

                poi['lat'] = poi.geometry.centroid.y
                poi['lon'] = poi.geometry.centroid.x
                poi['kategori'] = name

                df_result = poi[['lat', 'lon', 'kategori']].reset_index(drop=True)
                all_results[name] = df_result

                filename = f"data_titik_{name}_bandung.csv"
                df_result.to_csv(filename, index=False)
            else:
                print(f"Tidak ada data ditemukan untuk {name}")
        except Exception as e:
            print(f"Gagal mengunduh data untuk {name}: {e}")
            
    return all_results

download_results = get_poi_data(categories, polygon_bandung)

if download_results:
    df_final = pd.concat(download_results.values(), ignore_index=True)
    df_final.to_csv("data_titik_poi_lengkap_jabar.csv", index=False)
    print("\n" + "="*30)
    print("PROSES SELESAI!")
    print(f"Total data gabungan: {len(df_final)} baris.")
    print("File utama: data_titik_poi_lengkap_jabar.csv")



Lagi ngunduh data sekolah...


C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:29: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lat'] = poi.geometry.centroid.y
C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lon'] = poi.geometry.centroid.x


Lagi ngunduh data restoran...


C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:29: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lat'] = poi.geometry.centroid.y
C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lon'] = poi.geometry.centroid.x


Lagi ngunduh data pasar...


...

C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:29: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lat'] = poi.geometry.centroid.y
C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lon'] = poi.geometry.centroid.x


Lagi ngunduh data halte_bus...


C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:29: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lat'] = poi.geometry.centroid.y
C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lon'] = poi.geometry.centroid.x


Lagi ngunduh data kafe...

PROSES SELESAI!
Total data gabungan: 2117 baris.
File utama: data_titik_poi_lengkap_jabar.csv


C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:29: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lat'] = poi.geometry.centroid.y
C:\Users\Hilmi Mithwa\AppData\Local\Temp\ipykernel_23436\3710180064.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poi['lon'] = poi.geometry.centroid.x
